# Assignment 9 — Transformer NMT with Pretrained Embeddings (English → Hindi)
**ILLM** · **U23AI032**

Same data and the same length-binned split as Assignment 8 (BPCC `ilci`, English → Hindi;
the split code is identical and seeded, so the test set is the same). An OpenNMT-py
**Transformer** is trained twice, with two kinds of pretrained input embeddings:

1. **fastText** word vectors (`wiki.en.vec`, `wiki.hi.vec`, 300-d) on **word** tokens.
2. **BPEmb** subword vectors (`bpemb`, 10k BPE vocab, 300-d) — the text is first encoded into
   BPEmb subwords, then the matching BPEmb vectors initialise the embedding tables.

Both runs are scored with `sacrebleu` BLEU (plus chrF) **per length bin**.

Run on a GPU runtime (e.g. Colab T4). OpenNMT-py 3.5.1 pins an older `torch`; if the runtime
complains after the install, restart it once and re-run from the top.

In [ ]:
!pip install -q "OpenNMT-py==3.5.1" "numpy<2" sacrebleu bpemb requests huggingface_hub pandas matplotlib

## 0. Settings and imports

In [ ]:
import os, re, csv, glob

import numpy as np
import pandas as pd
import requests
import torch
import yaml

SEED = 42
TGT_LANG = "hin_Deva"            # native language = target, English = source
LANG2 = {"hin_Deva": "hi", "mar_Deva": "mr", "ben_Beng": "bn", "guj_Gujr": "gu",
         "tam_Taml": "ta", "tel_Telu": "te", "kan_Knda": "kn", "mal_Mlym": "ml",
         "pan_Guru": "pa", "urd_Arab": "ur", "npi_Deva": "ne", "asm_Beng": "as"}
TGT2 = LANG2[TGT_LANG]           # 2-letter code used by fastText and BPEmb

HERE = os.getcwd()
DATA_DIR = os.path.join(HERE, "mt_data")      # same files as Lab 8
FT_RUN = os.path.join(HERE, "lab9_fasttext")
BPE_RUN = os.path.join(HERE, "lab9_bpemb")
for d in (DATA_DIR, FT_RUN, BPE_RUN):
    os.makedirs(d, exist_ok=True)

BINS = ["bin1", "bin2", "bin3", "bin4"]
TEST_PER_BIN = 500
VALID_SIZE = 1000

DIM = 300                        # fastText and BPEmb vectors are both 300-d
BPE_VS = 10000
TRAIN_STEPS = 20000
FT_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.{}.vec"

USE_GPU = torch.cuda.is_available()
GPU_FLAG = "-gpu 0" if USE_GPU else ""
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if USE_GPU else "none")

## 1. Load the BPCC **ilci** subset (English → native language)

BPCC is a gated dataset: accept its terms at https://huggingface.co/datasets/ai4bharat/BPCC and
log in (`huggingface-cli login`, or set `HF_TOKEN`) before running this cell.

In [ ]:
from huggingface_hub import hf_hub_download

tsv_path = hf_hub_download("ai4bharat/BPCC", f"ilci/{TGT_LANG}.tsv", repo_type="dataset")
raw = pd.read_csv(tsv_path, sep="\t", quoting=csv.QUOTE_NONE, dtype=str,
                  keep_default_na=False, on_bad_lines="skip")
print("columns:", raw.columns.tolist(), "| rows:", len(raw))
raw.head()

In [ ]:
def latin_ratio(texts):
    s = "".join(texts)
    return sum(c.isascii() for c in s) / max(len(s), 1)


# the English column is the one written (mostly) in ASCII
cols = ["src", "tgt"] if {"src", "tgt"} <= set(raw.columns) else list(raw.columns[:2])
en_col = max(cols, key=lambda c: latin_ratio(raw[c].head(500)))
tgt_col = [c for c in cols if c != en_col][0]


def clean(s):
    return re.sub(r"\s+", " ", s).strip()


data = pd.DataFrame({"en": raw[en_col].map(clean), "tgt": raw[tgt_col].map(clean)})
data = data[(data.en != "") & (data.tgt != "")].drop_duplicates("en").reset_index(drop=True)
print(f"English column = {en_col!r}, {TGT_LANG} column = {tgt_col!r}")
print("clean sentence pairs:", len(data))
data.sample(5, random_state=SEED)

## 2. Length bins and the train / valid / test split

Sentence length = number of whitespace tokens in the **English (source)** sentence.
Lengths are integers, so the bins are read as:

| bin | length |
|-----|--------|
| bin1 | ≤ 10 |
| bin2 | 11 – 20 |
| bin3 | 21 – 29 |
| bin4 | ≥ 30 |

The split is **not** a plain random split: every bin is split separately, so both train and test
contain sentences from all four length ranges. The test set takes the same number of sentences
from each bin (`TEST_PER_BIN`, capped at 10 % of a bin), so every bin's BLEU is computed on a
comparably sized sample. A small validation set (for OpenNMT's periodic validation) is carved out
of the training part.

In [ ]:
def length_bin(n):
    if n <= 10:
        return "bin1"
    if n <= 20:
        return "bin2"
    if n < 30:
        return "bin3"
    return "bin4"


data["length"] = data.en.str.split().map(len)
data["bin"] = data.length.map(length_bin)

rng = np.random.default_rng(SEED)
train_parts, test_parts = [], []
for b in BINS:
    group = data[data.bin == b]
    group = group.iloc[rng.permutation(len(group))]
    n_test = min(TEST_PER_BIN, len(group) // 10)
    test_parts.append(group.iloc[:n_test])
    train_parts.append(group.iloc[n_test:])

train_all = pd.concat(train_parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
valid_set = train_all.iloc[:VALID_SIZE].reset_index(drop=True)
train_set = train_all.iloc[VALID_SIZE:].reset_index(drop=True)
test_set = pd.concat(test_parts).reset_index(drop=True)          # ordered bin1 → bin4

summary = pd.DataFrame({name: split.bin.value_counts().reindex(BINS)
                        for name, split in [("train", train_set), ("valid", valid_set),
                                            ("test", test_set)]})
summary.loc["total"] = summary.sum()
summary

In [ ]:
def write_lines(path, lines):
    with open(path, "w", encoding="utf-8") as f:
        for line in lines:
            f.write(line + "\n")


def read_lines(path):
    with open(path, encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]


for name, split in [("train", train_set), ("valid", valid_set), ("test", test_set)]:
    write_lines(os.path.join(DATA_DIR, f"{name}.en"), split.en)
    write_lines(os.path.join(DATA_DIR, f"{name}.tgt"), split.tgt)
write_lines(os.path.join(DATA_DIR, "test.bins"), test_set.bin)

test_bins = list(test_set.bin)
print(sorted(os.listdir(DATA_DIR)))

## 3. Shared Transformer configuration

Both runs use the same Transformer; only the tokenisation and the embedding files differ.
`hidden_size` must equal the embedding size, so the model width is 300 (6 heads × 50).

| setting | value |
|---------|-------|
| layers | 4 encoder + 4 decoder |
| d_model / heads / FFN | 300 / 6 / 1200 |
| positional encoding | sinusoidal |
| optimiser | Adam (β₂ = 0.998), Noam schedule, 4000 warm-up steps |
| regularisation | dropout 0.1, label smoothing 0.1 |
| batch | 4096 tokens, 20k steps |
| embeddings | pretrained (`word2vec` text format), fine-tuned during training |

In [ ]:
def transformer_config(run_dir, src_emb=None, tgt_emb=None, extra=None):
    cfg = {
        "save_data": os.path.join(run_dir, "data"),
        "src_vocab": os.path.join(run_dir, "vocab.src"),
        "tgt_vocab": os.path.join(run_dir, "vocab.tgt"),
        "overwrite": True,
        "data": {
            "corpus_1": {"path_src": os.path.join(run_dir, "train.tok.en"),
                         "path_tgt": os.path.join(run_dir, "train.tok.tgt"),
                         "transforms": ["filtertoolong"]},
            "valid": {"path_src": os.path.join(run_dir, "valid.tok.en"),
                      "path_tgt": os.path.join(run_dir, "valid.tok.tgt")},
        },
        "src_seq_length": 200,
        "tgt_seq_length": 200,

        # model
        "encoder_type": "transformer",
        "decoder_type": "transformer",
        "position_encoding": True,
        "enc_layers": 4,
        "dec_layers": 4,
        "heads": 6,
        "hidden_size": DIM,
        "word_vec_size": DIM,
        "transformer_ff": 1200,
        "dropout": [0.1],
        "attention_dropout": [0.1],

        # training
        "save_model": os.path.join(run_dir, "model"),
        "train_steps": TRAIN_STEPS,
        "valid_steps": 1000,
        "save_checkpoint_steps": 5000,
        "keep_checkpoint": 2,
        "report_every": 500,
        "batch_type": "tokens",
        "batch_size": 4096,
        "valid_batch_size": 2048,
        "optim": "adam",
        "adam_beta2": 0.998,
        "decay_method": "noam",
        "learning_rate": 2.0,
        "warmup_steps": 4000,
        "label_smoothing": 0.1,
        "max_grad_norm": 0,
        "param_init": 0,
        "param_init_glorot": True,
        "normalization": "tokens",
        "seed": SEED,
    }
    if src_emb:
        cfg.update({"src_embeddings": src_emb, "tgt_embeddings": tgt_emb,
                    "embeddings_type": "word2vec"})
    if USE_GPU:
        cfg.update({"world_size": 1, "gpu_ranks": [0]})
    cfg.update(extra or {})
    path = os.path.join(run_dir, "transformer.yaml")
    with open(path, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
    return path


def read_vocab(path):
    return {line.split("\t")[0] for line in read_lines(path) if line}


def write_vec(path, words, vectors):
    # word2vec text format: header "<count> <dim>", then one "word v1 ... vd" per line
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"{len(words)} {vectors.shape[1]}\n")
        for w, v in zip(words, vectors):
            f.write(w + " " + " ".join(f"{x:.5f}" for x in v) + "\n")


def latest_checkpoint(run_dir):
    ckpts = glob.glob(os.path.join(run_dir, "model_step_*.pt"))
    return max(ckpts, key=lambda p: int(re.search(r"step_(\d+)", p).group(1)))

In [ ]:
import sacrebleu


def binwise_scores(hyps, refs, bins):
    rows = []
    for b in BINS + ["all"]:
        idx = [i for i, x in enumerate(bins) if b == "all" or x == b]
        h = [hyps[i] for i in idx]
        r = [refs[i] for i in idx]
        bleu = sacrebleu.corpus_bleu(h, [r])
        chrf = sacrebleu.corpus_chrf(h, [r])
        rows.append({"bin": b, "sentences": len(idx), "BLEU": round(bleu.score, 2),
                     "chrF": round(chrf.score, 2), "BP": round(bleu.bp, 3),
                     "hyp/ref len": round(bleu.sys_len / max(bleu.ref_len, 1), 3),
                     "1/2/3/4-gram prec": "/".join(f"{p:.1f}" for p in bleu.precisions)})
    return pd.DataFrame(rows).set_index("bin")

---
## 4. Experiment A — Transformer + fastText word embeddings

### 4.1 Word tokenisation
fastText `.vec` files hold whole words, so the text is split into words, with punctuation
(including the Devanagari danda `।`) separated. English is lowercased to match more vectors;
Hindi has no case.

In [ ]:
PUNCT = re.compile(r'([.,!?;:"“”‘’()\[\]{}।॥|])')


def word_tokenize(s, lower=False):
    s = PUNCT.sub(r" \1 ", s.lower() if lower else s)
    return s.split()


for name in ["train", "valid", "test"]:
    en = read_lines(os.path.join(DATA_DIR, f"{name}.en"))
    tgt = read_lines(os.path.join(DATA_DIR, f"{name}.tgt"))
    write_lines(os.path.join(FT_RUN, f"{name}.tok.en"), [" ".join(word_tokenize(s, lower=True)) for s in en])
    write_lines(os.path.join(FT_RUN, f"{name}.tok.tgt"), [" ".join(word_tokenize(s)) for s in tgt])

print(read_lines(os.path.join(FT_RUN, "train.tok.en"))[0])
print(read_lines(os.path.join(FT_RUN, "train.tok.tgt"))[0])

In [ ]:
FT_CONFIG = transformer_config(FT_RUN, extra={"src_words_min_frequency": 2,
                                              "tgt_words_min_frequency": 2})
!onmt_build_vocab -config {FT_CONFIG} -n_sample -1

### 4.2 Pretrained fastText vectors, filtered to the vocabulary

`wiki.en.vec` is several GB, so it is **streamed** and only the vectors for words in our
OpenNMT vocabulary are kept (an exact match wins; otherwise a case-insensitive match is used).
The result is a small `word2vec`-format file that OpenNMT loads into the embedding tables.
Words without a fastText vector keep their random initialisation.

In [ ]:
def filtered_fasttext(lang, vocab, out_path):
    if os.path.exists(out_path):
        return read_lines(out_path)[0]
    found, exact = {}, set()
    with requests.get(FT_URL.format(lang), stream=True, timeout=120) as r:
        r.raise_for_status()
        lines = r.iter_lines()
        dim = int(next(lines).split()[1])
        for raw_line in lines:
            parts = raw_line.decode("utf-8", errors="ignore").rstrip().split(" ")
            if len(parts) != dim + 1:
                continue
            w = parts[0]
            if w in vocab:
                found[w] = parts[1:]
                exact.add(w)
            elif w.lower() in vocab and w.lower() not in exact and w.lower() not in found:
                found[w.lower()] = parts[1:]
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(f"{len(found)} {dim}\n")
        for w, vec in found.items():
            f.write(w + " " + " ".join(vec) + "\n")
    return f"{len(found)} {dim}"


ft_paths = {}
for side, lang in [("src", "en"), ("tgt", TGT2)]:
    vocab = read_vocab(os.path.join(FT_RUN, f"vocab.{side}"))
    ft_paths[side] = os.path.join(FT_RUN, f"fasttext.{side}.vec")
    print(f"streaming wiki.{lang}.vec ...")
    header = filtered_fasttext(lang, vocab, ft_paths[side])
    n_found = int(header.split()[0])
    print(f"  {side}: {n_found}/{len(vocab)} vocabulary words have a fastText vector "
          f"({100 * n_found / len(vocab):.1f} %)")

In [ ]:
FT_CONFIG = transformer_config(FT_RUN, src_emb=ft_paths["src"], tgt_emb=ft_paths["tgt"],
                               extra={"src_words_min_frequency": 2,
                                      "tgt_words_min_frequency": 2})
print(open(FT_CONFIG).read())

In [ ]:
!onmt_train -config {FT_CONFIG}

In [ ]:
FT_CKPT = latest_checkpoint(FT_RUN)
FT_SRC = os.path.join(FT_RUN, "test.tok.en")
FT_PRED = os.path.join(FT_RUN, "pred.tok.tgt")
print("checkpoint:", FT_CKPT)
!onmt_translate -model {FT_CKPT} -src {FT_SRC} -output {FT_PRED} -beam_size 5 -batch_size 64 -max_length 250 -replace_unk {GPU_FLAG}

### 4.3 Bin-wise BLEU — fastText
Hypotheses are word-tokenised, so the references go through the same `word_tokenize` before
scoring (sacrebleu's own 13a tokeniser is then applied to both in the same way).

In [ ]:
srcs = read_lines(os.path.join(DATA_DIR, "test.en"))
refs_raw = read_lines(os.path.join(DATA_DIR, "test.tgt"))

ft_hyps = read_lines(FT_PRED)
ft_refs = [" ".join(word_tokenize(s)) for s in refs_raw]
assert len(ft_hyps) == len(ft_refs) == len(test_bins)

ft_scores = binwise_scores(ft_hyps, ft_refs, test_bins)
ft_scores.to_csv(os.path.join(FT_RUN, "binwise_bleu.csv"))
ft_scores

---
## 5. Experiment B — Transformer + BPEmb subword embeddings

### 5.1 Encode the text into BPEmb subwords
As the BPEmb README describes, each sentence is first split with that language's BPEmb
SentencePiece model (`BPEmb.encode`). BPEmb lowercases and maps digits to `0` before encoding.

In [ ]:
from bpemb import BPEmb

bpemb = {"en": BPEmb(lang="en", vs=BPE_VS, dim=DIM),
         "tgt": BPEmb(lang=TGT2, vs=BPE_VS, dim=DIM)}
print(bpemb["en"].encode("Tourists visit the Taj Mahal in Agra every year."))
print(bpemb["tgt"].encode(read_lines(os.path.join(DATA_DIR, "train.tgt"))[0]))

In [ ]:
for name in ["train", "valid", "test"]:
    for side in ["en", "tgt"]:
        lines = read_lines(os.path.join(DATA_DIR, f"{name}.{side}"))
        write_lines(os.path.join(BPE_RUN, f"{name}.tok.{side}"),
                    [" ".join(bpemb[side].encode(s)) for s in lines])

print(read_lines(os.path.join(BPE_RUN, "train.tok.en"))[0])
print(read_lines(os.path.join(BPE_RUN, "train.tok.tgt"))[0])

### 5.2 Export the BPEmb vectors
The whole BPEmb table for each language (10k subwords × 300) is written in `word2vec` text format.
Because the corpus was segmented with the same BPE model, almost every subword in the OpenNMT
vocabulary gets a pretrained vector.

In [ ]:
bpe_paths = {}
for side, key in [("src", "en"), ("tgt", "tgt")]:
    kv = bpemb[key].emb
    words = list(getattr(kv, "index_to_key", None) or kv.index2word)
    bpe_paths[side] = os.path.join(BPE_RUN, f"bpemb.{side}.vec")
    write_vec(bpe_paths[side], words, kv.vectors)

BPE_CONFIG = transformer_config(BPE_RUN)
!onmt_build_vocab -config {BPE_CONFIG} -n_sample -1

In [ ]:
for side, key in [("src", "en"), ("tgt", "tgt")]:
    vocab = read_vocab(os.path.join(BPE_RUN, f"vocab.{side}"))
    kv = bpemb[key].emb
    known = set(getattr(kv, "index_to_key", None) or kv.index2word)
    print(f"{side}: {len(vocab & known)}/{len(vocab)} vocabulary subwords have a BPEmb vector")

BPE_CONFIG = transformer_config(BPE_RUN, src_emb=bpe_paths["src"], tgt_emb=bpe_paths["tgt"])
print(open(BPE_CONFIG).read())

In [ ]:
!onmt_train -config {BPE_CONFIG}

In [ ]:
BPE_CKPT = latest_checkpoint(BPE_RUN)
BPE_SRC = os.path.join(BPE_RUN, "test.tok.en")
BPE_PRED = os.path.join(BPE_RUN, "pred.tok.tgt")
print("checkpoint:", BPE_CKPT)
!onmt_translate -model {BPE_CKPT} -src {BPE_SRC} -output {BPE_PRED} -beam_size 5 -batch_size 64 -max_length 250 {GPU_FLAG}

### 5.3 Bin-wise BLEU — BPEmb
Outputs are decoded from subwords back to text with `BPEmb.decode`. The references are passed
through `encode → decode` as well, so they get the same lowercasing and digit normalisation and
the comparison is fair.

In [ ]:
bpe_hyps = [bpemb["tgt"].decode(line.split()) for line in read_lines(BPE_PRED)]
bpe_refs = [bpemb["tgt"].decode(bpemb["tgt"].encode(s)) for s in refs_raw]
assert len(bpe_hyps) == len(bpe_refs) == len(test_bins)

bpe_scores = binwise_scores(bpe_hyps, bpe_refs, test_bins)
bpe_scores.to_csv(os.path.join(BPE_RUN, "binwise_bleu.csv"))
bpe_scores

---
## 6. Comparison

In [ ]:
comparison = pd.DataFrame({
    "sentences": ft_scores["sentences"],
    "fastText BLEU": ft_scores["BLEU"],
    "BPEmb BLEU": bpe_scores["BLEU"],
    "fastText chrF": ft_scores["chrF"],
    "BPEmb chrF": bpe_scores["chrF"],
})

lab8_csv = os.path.join(HERE, "lab8_rnn", "binwise_bleu.csv")
if os.path.exists(lab8_csv):                       # add the Lab 8 LSTM baseline if it was run here
    lab8 = pd.read_csv(lab8_csv, index_col="bin")
    comparison.insert(1, "Lab 8 LSTM BLEU", lab8["BLEU"])
comparison

In [ ]:
import matplotlib.pyplot as plt

bleu_cols = [c for c in comparison.columns if c.endswith("BLEU")]
x = np.arange(len(BINS))
w = 0.8 / len(bleu_cols)
fig, ax = plt.subplots(figsize=(8, 4))
for k, col in enumerate(bleu_cols):
    vals = comparison.loc[BINS, col]
    ax.bar(x + (k - (len(bleu_cols) - 1) / 2) * w, vals, w, label=col.replace(" BLEU", ""))
ax.set_xticks(x, ["bin1\n≤10", "bin2\n11–20", "bin3\n21–29", "bin4\n≥30"])
ax.set_ylabel("BLEU")
ax.set_title("BLEU by source-length bin")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
for b in BINS:
    i = test_bins.index(b)
    print(f"--- {b} ---")
    print("SRC      :", srcs[i])
    print("REF      :", refs_raw[i])
    print("fastText :", ft_hyps[i])
    print("BPEmb    :", bpe_hyps[i])

## 7. Bin-wise analysis

*(Read together with the comparison table and plot above — the numbers come from the run.)*

**Transformer vs. the Lab 8 LSTM.** Self-attention connects every source position to every other
in one step, so the Transformer does not have to carry a long sentence through a recurrent state.
The gain should be largest in **bin3 and bin4**, where the LSTM's BLEU fell most. In short
sentences (bin1) all models are close, because there is little reordering or long-distance
agreement to get wrong.

**fastText (words) vs. BPEmb (subwords).**

* **Coverage / OOV.** With word tokens, every word not in the vocabulary (rare names, numbers,
  inflected forms of Hindi verbs and nouns) becomes `<unk>`. `-replace_unk` copies a source word
  through attention, but that only helps for names. BPEmb breaks any word into known subwords, so
  it has no OOVs and almost the whole vocabulary starts from a pretrained vector. Hindi
  morphology (gender/number/case endings such as -ा/-े/-ी, postpositions, verb auxiliaries) is
  modelled well by subwords, so **BPEmb usually scores higher, especially on chrF.**
* **Length bins.** OOVs pile up in long sentences (more content words, more names), so the gap
  between the two embeddings tends to grow from bin1 to bin4. In bin1, word-level fastText can be
  competitive: short ILCI sentences use common words that all have good fastText vectors.
* **What the pretrained vectors add.** ILCI is small (tens of thousands of pairs), so starting
  from pretrained vectors gives the model lexical similarity it could not learn from this data
  alone. The embeddings are fine-tuned, so the pretrained vectors act as an initialisation, not a
  fixed constraint.

**Caveat on comparing numbers.** The two runs are scored on slightly different text: the fastText
run is word-tokenised, and the BPEmb run is lowercased with digits normalised. BLEU is comparable
across bins *within* a run. Across runs, chrF and the general trend are more reliable than small
BLEU differences.